
---

## 📚 Sobre este Material

Este material ha sido diseñado con el propósito de **capacitar, actualizar y practicar** conceptos fundamentales de Markdown en Jupyter Notebook. Es una herramienta pensada para facilitar el aprendizaje y la documentación efectiva de proyectos de análisis de datos y ciencia de datos.

### 🤝 Compartir y Colaborar

Este contenido es **libre para compartir, revisar, divulgar y mejorar**. Se promueve activamente su distribución en la comunidad para que más personas puedan beneficiarse y contribuir a su mejora continua. Tu feedback y sugerencias son siempre bienvenidos.

### 👨‍💻 Autor

**Andrés Muñoz**  
*AI & Data Strategy Leader passionate about NLP, LLMs, and MLOps. Driving innovation with data*

- 💼 LinkedIn: [in/amms1989](https://linkedin.com/in/amms1989)
- 🐙 GitHub: [https://github.com/anguihero](https://github.com/anguihero)

---

# Sesión 13: Pipeline Completo de ML (End-to-End)

**Autor:** anmmunozsa@outlook.es · Material de código abierto para compartir y aprender colectivamente.

## 🎯 Objetivo de la sesión
Construir un pipeline reproducible que encapsule preprocesamiento + modelo en un solo objeto de scikit-learn, evitando el data leakage. Seguimos con `load_diabetes`.

## 🗺️ Tabla de Contenido
1. [Introducción: el problema de hacerlo "a mano"](#intro)
2. [El riesgo del Data Leakage](#leakage)
3. [Pipeline de scikit-learn](#pipeline)
4. [ColumnTransformer](#columntransformer)
5. [Pipeline + cross_val_score](#pipeline-cv)
6. [Reconstruyendo el "modelo campeón"](#campeon)
7. [Ejemplos de aplicación real](#aplicaciones)
8. [Retos de práctica](#retos)


<a id="intro"></a>
## 1. Introducción (para dummies)

En las Sesiones 07-12 hicimos el preprocesamiento "a mano": imputar, codificar, escalar, y luego entrenar. Eso funciona, pero es fácil cometer errores (olvidar un paso, aplicarlo en el orden incorrecto, o lo peor: filtrar información del conjunto de prueba). Un **Pipeline** empaqueta todos esos pasos en un solo objeto que se comporta como un modelo más: tiene `.fit()`, `.predict()` y se puede validar con `cross_val_score` sin riesgo de errores.

<a id="leakage"></a>
## 2. El Riesgo del Data Leakage

### 🔬 Teoría técnica
**Data leakage** ocurre cuando información del conjunto de prueba (o de datos futuros) se filtra al entrenamiento, dando métricas artificialmente optimistas que no se replican en producción.

**Ejemplo típico de error:** escalar (`fit_transform`) **todo** el dataset antes de hacer `train_test_split`. El escalador "vio" los datos de prueba al calcular la media/desviación, aunque se supone que no debería conocerlos.

In [ ]:
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import pandas as pd

datos = load_diabetes(as_frame=True)
X = datos.data
y = datos.target

# INCORRECTO (ejemplo educativo de qué NO hacer): escalar antes del split
escalador_incorrecto = StandardScaler()
X_escalado_completo = escalador_incorrecto.fit_transform(X)  # ve TODO el dataset, incluida la futura "prueba"

# CORRECTO: dividir primero, ajustar el escalador SOLO con entrenamiento
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
escalador_correcto = StandardScaler()
X_train_esc = escalador_correcto.fit_transform(X_train)   # fit_transform SOLO en train
X_test_esc = escalador_correcto.transform(X_test)          # transform (sin fit) en test

print("Esto es exactamente lo que Pipeline automatiza y garantiza siempre.")

<a id="pipeline"></a>
## 3. Pipeline de scikit-learn

### 🔬 Teoría técnica
`Pipeline` encadena una lista de pasos `(nombre, transformador_o_modelo)`. Al llamar `.fit()`, cada paso hace `fit_transform` en orden, y el último paso (el modelo) hace `fit` normal. Al llamar `.predict()`, cada paso hace solo `transform`.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Ridge

pipeline_simple = Pipeline(steps=[
    ("escalador", StandardScaler()),
    ("modelo", Ridge(alpha=1.0)),
])

pipeline_simple.fit(X_train, y_train)
print("R² en prueba:", pipeline_simple.score(X_test, y_test))

### 🧠 Resumen para dummies
`Pipeline` es como una línea de ensamblaje: los datos entran por un extremo, pasan por cada estación (escalado, imputación, etc.) en orden, y salen convertidos en una predicción.

<a id="columntransformer"></a>
## 4. ColumnTransformer

### 🔬 Teoría técnica
Cuando el dataset mezcla columnas numéricas y categóricas, cada tipo necesita un tratamiento distinto (`StandardScaler` para numéricas, `OneHotEncoder` para categóricas). `ColumnTransformer` aplica cada transformación **solo a las columnas indicadas**, y luego junta todo en una sola matriz.

`load_diabetes` es 100% numérico, así que aquí mostramos la rama numérica. La sintaxis se **extendería** exactamente así si tuviéramos columnas categóricas (como en `application_data.csv`, Sesión 07):

```python
ColumnTransformer([
    ("num", StandardScaler(), columnas_numericas),
    ("cat", OneHotEncoder(handle_unknown="ignore"), columnas_categoricas),
])
```

In [ ]:
from sklearn.compose import ColumnTransformer

columnas_numericas = X.columns.tolist()  # en load_diabetes, todas son numéricas

preprocesador = ColumnTransformer(transformers=[
    ("num", StandardScaler(), columnas_numericas),
    # ("cat", OneHotEncoder(handle_unknown="ignore"), columnas_categoricas),  # así se vería con categóricas
])

pipeline_ct = Pipeline(steps=[
    ("preprocesador", preprocesador),
    ("modelo", Ridge(alpha=1.0)),
])

pipeline_ct.fit(X_train, y_train)
print("R² en prueba:", pipeline_ct.score(X_test, y_test))

### 🧠 Resumen para dummies
`ColumnTransformer` es un "clasificador de equipaje": manda cada columna al tratamiento correcto según su tipo, y al final junta todo en una sola tabla lista para el modelo.

<a id="pipeline-cv"></a>
## 5. Pipeline + cross_val_score

### 🔬 Teoría técnica
Al pasar el `Pipeline` completo a `cross_val_score`, scikit-learn recalcula el escalado (y cualquier imputación) **desde cero en cada fold**, usando solo los datos de entrenamiento de ese fold — el data leakage queda prevenido automáticamente.

In [ ]:
from sklearn.model_selection import cross_val_score

scores = cross_val_score(pipeline_ct, X, y, cv=5, scoring="r2")
print(f"R² promedio (5-fold): {scores.mean():.3f} +/- {scores.std():.3f}")

<a id="campeon"></a>
## 6. Reconstruyendo el "Modelo Campeón" como Pipeline

Tomamos el mejor resultado de la Sesión 08 (Regresión Lineal/Ridge) y lo formalizamos como un único `Pipeline`, listo para guardar y reutilizar.

In [ ]:
import joblib

pipeline_final = Pipeline(steps=[
    ("preprocesador", ColumnTransformer([("num", StandardScaler(), columnas_numericas)])),
    ("modelo", Ridge(alpha=1.0)),
])

pipeline_final.fit(X_train, y_train)

# Guardar el pipeline completo (preprocesamiento + modelo) en un solo archivo
joblib.dump(pipeline_final, "pipeline_diabetes.joblib")

# Cargarlo de vuelta y usarlo, exactamente como en producción
pipeline_cargado = joblib.load("pipeline_diabetes.joblib")
print("R² del pipeline cargado:", pipeline_cargado.score(X_test, y_test))

### 🧠 Resumen para dummies
Guardar un `Pipeline` completo (no solo el modelo) garantiza que en producción los datos nuevos pasen exactamente por los mismos pasos de preprocesamiento que se usaron en el entrenamiento.

## 🔎 Laboratorio de profundización: contrato fit/transform/predict

Un estimador de scikit-learn sigue una API:

- `fit(X, y)` aprende parámetros.
- `transform(X)` aplica una transformación.
- `predict(X)` produce una salida.
- `get_params()` expone hiperparámetros.
- atributos con `_` guardan propiedades aprendidas.

Esto permite encadenar piezas y optimizar parámetros anidados con `paso__parametro`.


In [ ]:
# Paso 1: inspeccionar nombres de hiperparámetros del pipeline
parametros_pipeline = pipeline_final.get_params()
for nombre in sorted(k for k in parametros_pipeline if "__" in k)[:12]:
    print(nombre, "=", parametros_pipeline[nombre])


In [ ]:
# Paso 2: cambiar configuración sin reconstruir el objeto
pipeline_variante = pipeline_final.set_params(modelo__alpha=10.0)
pipeline_variante.fit(X_train, y_train)
print("Alpha usado:", pipeline_variante.named_steps["modelo"].alpha)
print("R² prueba:", pipeline_variante.score(X_test, y_test))


In [ ]:
# Paso 3: inspeccionar propiedades aprendidas
preprocesador_ajustado = pipeline_variante.named_steps["preprocesador"]
modelo_ajustado = pipeline_variante.named_steps["modelo"]
print("Transformadores:", preprocesador_ajustado.transformers_)
print("Primeros coeficientes:", modelo_ajustado.coef_[:5])


### Extensión a tipos mixtos

Una rama numérica puede usar `SimpleImputer(strategy="median")` + `StandardScaler`; una categórica, `SimpleImputer(strategy="most_frequent")` + `OneHotEncoder(handle_unknown="ignore")`. `remainder="drop"` descarta columnas no listadas; `remainder="passthrough"` las conserva. La lista de columnas es también parte del contrato reproducible.


<a id="aplicaciones"></a>
## 7. Ejemplos de Aplicación en el Mundo Real

- En la industria, casi nunca se preprocesa "a mano" fuera de un pipeline: se pierde reproducibilidad y aumenta el riesgo de errores.
- Un pipeline guardado (`joblib`) es lo que normalmente se despliega detrás de una API de predicción en producción.

<a id="retos"></a>
## 8. Retos de Práctica

### 🥉 Reto Básico
Construye un `Pipeline` de 2 pasos (`StandardScaler` + `LinearRegression`) sobre `load_diabetes` y evalúa su R² en el conjunto de prueba.

In [ ]:
# Tu solución al Reto Básico aquí


### 🥈 Reto Medio
Construye un `Pipeline` con `ColumnTransformer` (rama numérica) + `RandomForestRegressor`, validado con `cross_val_score` (5-fold, `scoring="r2"`). Compara el resultado contra el `pipeline_ct` con Ridge.

In [ ]:
# Tu solución al Reto Medio aquí


### 🥇 Reto Avanzado
Reconstruye como un único `Pipeline` el mejor modelo de regularización (Ridge o Lasso) con su mejor `alpha` encontrado en la Sesión 08, valídalo con `cross_val_score`, y documenta en una celda de Markdown cómo se extendería el `ColumnTransformer` de este pipeline si `load_diabetes` tuviera columnas categóricas como las de `application_data.csv` (Sesión 07).

In [ ]:
# Tu solución al Reto Avanzado aquí
